# 技能0 · Day 2 上机：营销数据结构实战

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **list** 对营销订单列表做排序、筛选、切片、提取唯一客户ID
2. 用 **dict** 构建产品目录映射，做 O(1) 查询和按类别筛选
3. 用 **set** 做用户标签运算（交集/并集/差集/对称差集）和跨渠道用户去重
4. 用 **Counter/defaultdict** 统计商品销量排行、按渠道分组订单、按用户聚合行为
5. 用 **deque** 模拟用户浏览路径（maxlen滑动窗口）和订单处理队列
6. 用 **namedtuple** 设计 Product 数据类，用 dict 构建产品分类树并 BFS 遍历

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：Python 内置（list/dict/set/tuple）+ collections（Counter/defaultdict/namedtuple/deque）+ heapq。
营销映射：营销订单数据（订单列表 + 产品目录 + 客户画像 + 用户标签 + 行为序列）。

## 0. 环境准备
本 Day 使用 Python 标准库，无需额外安装。如需扩展对比可取消注释安装 pandas/Polars。

> ⚠️ Python 3.8+ 即可运行全部代码。collections 和 heapq 是标准库，无需 pip install。
> 数据治理提示：用 namedtuple 定义 schema 是数据规范化的第一步，确保字段一致性可追溯。

In [ ]:
# 本 Day 使用 Python 标准库，通常无需额外安装
# 如需扩展（pandas/Polars 对比）可取消注释：
# !pip install pandas polars pyarrow -q

## 1. 数据集背景与营销映射

**处理对象**：一组营销场景的真实数据（订单 + 产品 + 客户 + 标签 + 行为）。

| 数据 | 类型 | 说明 |
|------|------|------|
| orders | list of dicts | 15条订单，含金额/渠道/状态 |
| products | dict | 8个产品，含名称/类别/价格/库存 |
| customers | dict | 6个客户，含等级/区域/年龄 |
| user_tags | dict of sets | 每客户3-5个行为标签 |
| user_behaviors | list of tuples | 16条行为记录（浏览/点击/购买） |
| browsing_sequence | list of tuples | 用户浏览路径序列（8步） |

**营销映射**：在真实项目中，这些数据来自 CRM 系统、订单数据库、用户行为日志。数据结构的选择直接决定查询性能--用 dict 做 O(1) 产品查询 vs list 做 O(n) 线性查找，在 10 万条数据时差距是 10 万倍。

**理论连接**：Python 内置数据结构是 pandas/numpy/SQLAlchemy 等所有数据科学库的底层基础。理解 list/dict/set 的性能特性是理解 Apache Arrow 列式内存和 Polars 懒求值优势的前提。pandas 的 DataFrame 本质是 dict of numpy arrays。

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from collections import Counter, defaultdict, deque, namedtuple
import heapq

# ============================================================
# 营销数据集（内嵌，模拟真实电商数据）
# ============================================================

# 1. 订单列表（list of dicts）- 15条订单
orders = [
    {"order_id": "ORD001", "customer_id": "C001", "product_id": "P101", "amount": 1299.00, "date": "2025-06-15", "channel": "app", "status": "completed"},
    {"order_id": "ORD002", "customer_id": "C002", "product_id": "P103", "amount": 89.00, "date": "2025-06-16", "channel": "web", "status": "completed"},
    {"order_id": "ORD003", "customer_id": "C003", "product_id": "P102", "amount": 599.00, "date": "2025-06-17", "channel": "app", "status": "completed"},
    {"order_id": "ORD004", "customer_id": "C001", "product_id": "P105", "amount": 159.00, "date": "2025-06-18", "channel": "mini_program", "status": "pending"},
    {"order_id": "ORD005", "customer_id": "C004", "product_id": "P101", "amount": 1299.00, "date": "2025-06-19", "channel": "app", "status": "completed"},
    {"order_id": "ORD006", "customer_id": "C002", "product_id": "P106", "amount": 39.00, "date": "2025-06-20", "channel": "web", "status": "cancelled"},
    {"order_id": "ORD007", "customer_id": "C005", "product_id": "P104", "amount": 4599.00, "date": "2025-06-21", "channel": "store", "status": "completed"},
    {"order_id": "ORD008", "customer_id": "C003", "product_id": "P107", "amount": 299.00, "date": "2025-06-22", "channel": "app", "status": "completed"},
    {"order_id": "ORD009", "customer_id": "C004", "product_id": "P102", "amount": 599.00, "date": "2025-06-23", "channel": "web", "status": "refunded"},
    {"order_id": "ORD010", "customer_id": "C006", "product_id": "P108", "amount": 129.00, "date": "2025-06-24", "channel": "mini_program", "status": "completed"},
    {"order_id": "ORD011", "customer_id": "C001", "product_id": "P103", "amount": 89.00, "date": "2025-06-25", "channel": "app", "status": "completed"},
    {"order_id": "ORD012", "customer_id": "C005", "product_id": "P106", "amount": 39.00, "date": "2025-06-26", "channel": "store", "status": "completed"},
    {"order_id": "ORD013", "customer_id": "C002", "product_id": "P101", "amount": 1299.00, "date": "2025-06-27", "channel": "app", "status": "completed"},
    {"order_id": "ORD014", "customer_id": "C006", "product_id": "P105", "amount": 159.00, "date": "2025-06-28", "channel": "web", "status": "pending"},
    {"order_id": "ORD015", "customer_id": "C003", "product_id": "P108", "amount": 129.00, "date": "2025-06-29", "channel": "mini_program", "status": "completed"},
]

# 2. 产品目录（dict）- product_id -> {name, category, price, stock}
products = {
    "P101": {"name": "智能手表Pro", "category": "electronics", "price": 1299.00, "stock": 50},
    "P102": {"name": "无线降噪耳机", "category": "electronics", "price": 599.00, "stock": 120},
    "P103": {"name": "烟酰胺精华液", "category": "skincare", "price": 89.00, "stock": 500},
    "P104": {"name": "健身跑步机", "category": "fitness", "price": 4599.00, "stock": 15},
    "P105": {"name": "瑜伽垫premium", "category": "fitness", "price": 159.00, "stock": 200},
    "P106": {"name": "蛋白粉1kg", "category": "fitness", "price": 39.00, "stock": 800},
    "P107": {"name": "电动牙刷", "category": "electronics", "price": 299.00, "stock": 300},
    "P108": {"name": "防晒霜SPF50", "category": "skincare", "price": 129.00, "stock": 400},
}

# 3. 客户画像（dict）- customer_id -> {name, level, region, age}
customers = {
    "C001": {"name": "张明", "level": "gold", "region": "华北", "age": 32},
    "C002": {"name": "李华", "level": "silver", "region": "华东", "age": 28},
    "C003": {"name": "王芳", "level": "gold", "region": "华南", "age": 35},
    "C004": {"name": "赵强", "level": "normal", "region": "华北", "age": 24},
    "C005": {"name": "陈静", "level": "gold", "region": "华东", "age": 45},
    "C006": {"name": "刘伟", "level": "silver", "region": "华南", "age": 30},
}

# 4. 用户标签（dict of sets）- customer_id -> set of tags
user_tags = {
    "C001": {"高频", "电子", "高消费", "APP用户", "VIP"},
    "C002": {"中频", "护肤", "中消费", "WEB用户"},
    "C003": {"高频", "电子", "高消费", "APP用户"},
    "C004": {"低频", "健身", "低消费", "小程序用户"},
    "C005": {"高频", "健身", "高消费", "门店用户", "VIP"},
    "C006": {"中频", "护肤", "中消费", "WEB用户"},
}

# 5. 用户行为序列（list of tuples）- (customer_id, action, product_id, timestamp)
user_behaviors = [
    ("C001", "view", "P101", "2025-06-15T09:00"),
    ("C001", "click", "P101", "2025-06-15T09:05"),
    ("C001", "view", "P102", "2025-06-15T09:10"),
    ("C001", "purchase", "P101", "2025-06-15T10:00"),
    ("C002", "view", "P103", "2025-06-16T14:00"),
    ("C002", "click", "P103", "2025-06-16T14:10"),
    ("C002", "purchase", "P103", "2025-06-16T15:00"),
    ("C003", "view", "P102", "2025-06-17T11:00"),
    ("C003", "click", "P102", "2025-06-17T11:05"),
    ("C003", "purchase", "P102", "2025-06-17T12:00"),
    ("C001", "view", "P105", "2025-06-18T16:00"),
    ("C004", "view", "P101", "2025-06-19T10:00"),
    ("C004", "click", "P101", "2025-06-19T10:05"),
    ("C004", "purchase", "P101", "2025-06-19T11:00"),
    ("C005", "view", "P104", "2025-06-21T13:00"),
    ("C005", "purchase", "P104", "2025-06-21T14:00"),
]

# 6. 用户浏览路径序列（用于 deque 模拟）- (customer_id, product_id)
browsing_sequence = [
    ("C001", "P101"), ("C001", "P102"), ("C001", "P103"),
    ("C001", "P105"), ("C001", "P107"), ("C001", "P108"),
    ("C001", "P104"), ("C001", "P106"),
]

print("营销数据集加载完成")
print(f"  订单数: {len(orders)}")
print(f"  产品数: {len(products)}")
print(f"  客户数: {len(customers)}")
print(f"  用户标签: {len(user_tags)} 个客户")
print(f"  行为序列: {len(user_behaviors)} 条记录")

## 1：用 list 做订单排序/筛选/切片

In [ ]:
# 1. 用 list 对营销订单列表做排序、筛选、切片、提取唯一客户ID
sorted_orders = sorted(orders, key=lambda o: o['amount'], reverse=True)
high_value_orders = [o for o in orders if o['status'] == 'completed' and o['amount'] > 500]
top5_orders = sorted_orders[:5]
top5_customers = {o['customer_id'] for o in top5_orders}

print("按金额降序前3条：")
for o in sorted_orders[:3]:
    print(f"  {o['order_id']} | {o['customer_id']} | ¥{o['amount']:.2f} | {o['channel']}")
print(f"\n高价值订单数: {len(high_value_orders)}")
print(f"Top 5 订单总金额: ¥{sum(o['amount'] for o in top5_orders):.2f}")
print(f"Top 5 涉及客户: {top5_customers}")

## 2. 数据结构理论基础回顾

### Python 五大内置数据结构

| 结构 | 有序 | 可变 | 查找 | 营销应用 |
|------|:----:|:----:|:----:|---------|
| list | ✅ | ✅ | O(n) | 订单排序/筛选/切片 |
| dict | ✅ | ✅ | O(1) | 产品目录映射 |
| set | ❌ | ✅ | O(1) | 用户去重/标签运算 |
| tuple | ✅ | ❌ | O(n) | 不可变记录 |
| deque | ✅ | ✅ | O(1)两端 | 浏览路径/处理队列 |

**核心洞察**：dict/set 的 O(1) 哈希查找源于哈希表实现。用 list 做产品查询是 O(n)，换成 dict 就是 O(1)--10 万条数据时差距是 10 万倍。这是数据治理中"数据结构规范化"的工程基础。

### collections 模块三大利器

- **Counter**：自动计数 + `most_common(n)` -> 商品销量排行
- **defaultdict**：自动初始化缺失键 -> 按渠道分组、按用户聚合
- **namedtuple**：不可变 + 字段名访问 -> Product/Order 数据类（schema 规范化）

### 与 pandas/Arrow 的连接

pandas DataFrame 本质是 dict of numpy arrays（列式存储的 Python 实现）。Apache Arrow 将这一概念升级为跨语言列式内存格式，实现零拷贝数据传递。Polars 基于 Arrow 实现懒求值，查询图是 DAG 数据结构。理解原生数据结构是理解这些高级库的前提。

## 2：用 dict 做产品目录映射

In [ ]:
# 2. 用 dict 构建产品目录映射，做查询和按类别筛选
p101_info = products.get("P101")
electronics = {pid: info for pid, info in products.items() if info['category'] == 'electronics'}
product_names = {pid: info['name'] for pid, info in products.items()}
category_counts = {}
for info in products.values():
    cat = info['category']
    category_counts[cat] = category_counts.get(cat, 0) + 1

print(f"P101 产品信息: {p101_info}")
print(f"\n电子产品 ({len(electronics)}个):")
for pid, info in electronics.items():
    print(f"  {pid}: {info['name']} - ¥{info['price']:.2f}")
print(f"\n产品名映射: {product_names}")
print(f"品类产品数: {category_counts}")

## 3：用 set 做用户标签运算

In [ ]:
# 3. 用 set 做用户标签运算和跨渠道用户去重
common_tags = user_tags["C001"] & user_tags["C002"]
all_tags = user_tags["C001"] | user_tags["C002"]
unique_c001 = user_tags["C001"] - user_tags["C002"]
different_tags = user_tags["C001"] ^ user_tags["C002"]

app_users = {o['customer_id'] for o in orders if o['channel'] == 'app'}
web_users = {o['customer_id'] for o in orders if o['channel'] == 'web'}
all_channel_users = app_users | web_users

print(f"C001 标签: {user_tags['C001']}")
print(f"C002 标签: {user_tags['C002']}")
print(f"\n共同标签（交集）: {common_tags}")
print(f"所有标签（并集）: {all_tags}")
print(f"C001 独有（差集）: {unique_c001}")
print(f"不同标签（对称差集）: {different_tags}")
print(f"\nAPP用户: {app_users}")
print(f"WEB用户: {web_users}")
print(f"去重总用户数: {len(all_channel_users)}")

## 3. Counter 与 defaultdict：数据聚合利器

### Counter：自动计数

`Counter` 是 dict 的子类，自动统计元素出现次数：
```python
Counter(o['product_id'] for o in orders if o['status'] == 'completed')
# -> Counter({'P101': 3, 'P108': 2, ...})
counter.most_common(3)  # -> Top 3 热销商品
```

### defaultdict：自动初始化

`defaultdict(list)` 在访问不存在的键时自动创建空列表，避免 KeyError：
```python
channel_orders = defaultdict(list)
for o in orders:
    channel_orders[o['channel']].append(o)
# -> {'app': [...], 'web': [...], ...}
```

### 性能对比

手写循环计数 vs Counter：Counter 用 C 实现计数逻辑，比纯 Python 循环快 2-5 倍。defaultdict 省去了 `if key not in dict: dict[key] = []` 的样板代码，更简洁且不易出错。

**可复现研究提示**：Counter 和 defaultdict 的迭代顺序在 Python 3.7+ 是插入顺序，确保数据处理管线可复现。

## 4：用 Counter/defaultdict 做销量统计

In [ ]:
# 4. 用 Counter 统计商品销量排行，用 defaultdict 按渠道分组订单
product_sales = Counter(o['product_id'] for o in orders if o['status'] == 'completed')
top3_products = product_sales.most_common(3)

channel_orders = defaultdict(list)
for o in orders:
    channel_orders[o['channel']].append(o)

user_actions = defaultdict(lambda: defaultdict(list))
for cust_id, action, prod_id, ts in user_behaviors:
    user_actions[cust_id][action].append((prod_id, ts))

print("商品销量排行（完成订单）:")
for pid, count in product_sales.most_common():
    name = products[pid]['name']
    print(f"  {pid} {name}: {count}单")
print(f"\nTop 3 热销商品: {[(products[p[0]]['name'], p[1]) for p in top3_products]}")
print(f"\n各渠道订单数:")
for ch, odrs in channel_orders.items():
    print(f"  {ch}: {len(odrs)}单, 总额¥{sum(o['amount'] for o in odrs):.2f}")
print(f"\nC001 行为聚合:")
for action, items in user_actions['C001'].items():
    print(f"  {action}: {items}")

## 5：用 deque 做浏览路径/处理队列

In [ ]:
# 5. 用 deque 模拟用户浏览路径（maxlen滑动窗口）和订单处理队列
recent_views = deque(maxlen=5)
for cust_id, prod_id in browsing_sequence[:6]:
    recent_views.append(prod_id)

current_path = list(recent_views)
undo_item = recent_views.pop()

pending_orders = [o for o in orders if o['status'] == 'pending']
pending_queue = deque(pending_orders)
processed = pending_queue.popleft()

print(f"浏览路径（加了6个,maxlen=5）: {current_path}")
print(f"撤销最后一步: 移除了 {undo_item}")
print(f"撤销后路径: {list(recent_views)}")
print(f"\nPending订单处理:")
print(f"  处理了: {processed['order_id']}")
print(f"  剩余: {[o['order_id'] for o in pending_queue]}")

## 4. 自定义数据结构：namedtuple 与树

### namedtuple：不可变数据类

`namedtuple` 创建类似类的不可变数据结构，比裸 dict 更规范：
```python
Product = namedtuple('Product', ['product_id', 'name', 'category', 'price'])
p = Product('P101', '智能手表Pro', 'electronics', 1299.00)
p.name  # 字段名访问，比 dict['name'] 更可读
```

**数据治理优势**：namedtuple 定义了 schema（字段名和顺序），确保所有 Product 对象字段一致。不可变性防止意外修改，支持可复现研究。Python 3.7+ 的 dataclasses 是更灵活的替代方案。

### 树结构：产品分类树

用嵌套 dict 表示树：
```python
tree = {"root": {"electronics": {"P101": "智能手表Pro", ...}, ...}}
```

BFS（广度优先）遍历用 deque 实现：每层节点入队，逐层处理。这与 Polars 的查询图（DAG）遍历同理--树/图遍历是数据结构的基础操作。

## 6：用 namedtuple+dict 做数据类/分类树

In [ ]:
# 6. 用 namedtuple 设计 Product 数据类，用 dict 构建产品分类树并BFS遍历
Product = namedtuple('Product', ['product_id', 'name', 'category', 'price'])

product_objects = [
    Product(pid, info['name'], info['category'], info['price'])
    for pid, info in products.items()
]

# 构建产品分类树: root -> categories -> products
category_tree = {"root": {}}
for p in product_objects:
    if p.category not in category_tree["root"]:
        category_tree["root"][p.category] = {}
    category_tree["root"][p.category][p.product_id] = p.name

# BFS 遍历树
tree_nodes_bfs = []
queue = deque([("root", 0, category_tree["root"])])
while queue:
    node_name, depth, node_val = queue.popleft()
    tree_nodes_bfs.append((node_name, depth))
    if isinstance(node_val, dict):
        for child_name, child_val in node_val.items():
            queue.append((child_name, depth + 1, child_val))

print(f"Product 类型: {Product}")
print(f"\n前3个Product对象:")
for p in product_objects[:3]:
    print(f"  {p}")
print(f"\n产品分类树（BFS遍历）:")
for node, depth in tree_nodes_bfs:
    indent = "  " * depth
    print(f"{indent}{node}")

## 5. 反思与前沿

### 反思问题
1. 用 list 和 dict 分别做产品查询，在 15 条数据时性能差异明显吗？在 10 万条时呢？（用 `%timeit` 实测）
2. Counter 的 `most_common(3)` 和 `sorted(counter.items(), key=lambda x: x[1], reverse=True)[:3]` 效果相同，但底层实现有何不同？
3. deque(maxlen=5) 的滑动窗口在用户浏览路径模拟中，比 list 的切片 `path[-5:]` 有什么优势？
4. namedtuple 相比裸 dict，在数据治理和可复现研究方面有什么具体好处？

### 2026 前沿：Apache Arrow + Polars

**Apache Arrow**（Apache 软件基金会）定义了语言无关的列式内存格式，实现 Python/R/Spark/Polars 间的零拷贝数据共享。list of dicts 是行式存储的直觉代表，Arrow 的列式格式是其性能优化版--分析单列时只读取该列内存，比行式存储快 10-100 倍。

**Polars**（Ritchie Vink, 2020）基于 Arrow 实现懒求值（lazy evaluation）：操作构建查询图（DAG），最后 `.collect()` 时全局优化后一次执行。查询图本质是 DAG 数据结构，与本 Day TODO6 的产品分类树同为树/图结构应用。

**数据治理与可复现**：用 namedtuple/dataclass 定义 schema 是数据规范化的第一步。可复现研究要求固定随机种子、记录数据版本、用不可变数据结构确保管线可追溯。OSF（Open Science Framework）提供数据版本管理平台。

> ⚠️ 数据结构选择影响 AI pipeline 性能。dict 的 O(1) 查找是索引操作的基础，Arrow 的列式格式是大规模分析的基础，两者一脉相承。

参考 [Apache Arrow](https://arrow.apache.org/docs/python/) + [Polars](https://pola.rs/) + [Python collections](https://docs.python.org/3/library/collections.html)。